## 0. Setup & configuration

Every value you might need to edit — file path, sheet names, the reporting
lag — lives here, in one place, at the top. Two reasons: (1) you shouldn't
have to hunt through the notebook to find the one line to change when a
sheet gets renamed, and (2) Jupyter cells don't have to be run top-to-bottom
— nothing stops you from running a later cell before an earlier one, or
re-running one cell after a kernel restart without re-running the rest. If
`SECTOR_SHEET` were defined down in section 3 instead, running section 4
without first running section 3 in the same kernel session would raise
`NameError: name 'SECTOR_SHEET' is not defined`. Keeping config at the top
means it exists the moment the kernel starts, before any loading logic can
possibly ask for it.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data_loader import load_field_long, load_sector_map, ffill_with_reporting_lag
from src.factors import momentum_12_1, value_factor, quality_factor, low_vol_factor, zscore_within_sector
from src.weighting import (
    compute_forward_return,
    compute_ic_series,
    compute_ic_weights,
    combine_weighted,
    kalman_filter_series,
    compute_factor_return_series,
    rolling_covariance_entries,
    build_cov_matrix,
    mean_variance_weights,
    shrink_covariance,
)
from src.portfolio import (
    quintile_top_holdings,
    quintile_bottom_holdings,
    compute_portfolio_returns,
    compute_long_short_returns,
    compute_benchmark_returns,
    compute_turnover_by_period,
    compute_long_short_turnover,
)
from src.backtest import performance_metrics, bootstrap_metric_distribution

DATA_PATH = Path("data/Dataset.xlsx")

# sheet names confirmed from the "Price"/"ROE"/"EVtoEBITDA"/"MarketCap"/
# "SharesOutstanding" values sheets seen in section 1 — adjust as needed.
FIELD_SHEETS = {
    "price": "Price",
    "roe": "ROE",
    "ev_ebitda": "EVtoEBITDA",
    "market_cap": "MarketCap",
    "shares_outstanding": "SharesOutstanding",
    "debt_equity": "DebtToEquity",  # placeholder name — fix or remove if not present
}
SECTOR_SHEET = "GICS"

REPORTING_LAG_DAYS = 45  # <- tune to your data source's actual reporting lag

MARKET_CAP_FLOOR = 500  # $ millions, matching the MarketCap sheet's units

IC_WINDOW = 12  # months of trailing IC history averaged into each factor's rolling weight

KALMAN_Q_TO_R_RATIO = 0.05  # process noise as a fraction of observation noise;
                             # smaller = more smoothing / slower to react to new IC,
                             # larger = filter tracks recent IC more closely

REBALANCE_MONTHS = [3, 6, 9, 12]  # calendar quarter-end months

TRANSACTION_COST_BPS = 10  # basis points per unit of turnover, per rebalance

PERIODS_PER_YEAR = 4  # quarterly rebalancing, for annualizing performance metrics
RISK_FREE_RATE = 0.0  # simplifying assumption -- a real Sharpe would subtract a T-bill rate

COV_SHRINKAGE = 0.3  # 0 = no shrinkage, 1 = fully diagonal (factors treated as uncorrelated)

BLOCK_LENGTH = 4   # quarters per resampled block in the bootstrap (~1 year)
N_BOOTSTRAP = 2000  # number of bootstrap resamples


## 1. Inspect the workbook

Bloomberg-style exports often include a live "`- Formulas`" sheet next to a
pasted-values sheet for the same field. The formula sheets show `#N/A` once
opened outside a Bloomberg terminal, so we want the plain-values sheets.
Run this first and confirm the sheet names below match what's actually in
`Dataset.xlsx` before trusting the loader in the next section.

In [43]:
xls = pd.ExcelFile(DATA_PATH)
print(xls.sheet_names)

for name in xls.sheet_names:
    preview = pd.read_excel(DATA_PATH, sheet_name=name, nrows=3)
    print(f"\n=== {name} ===  shape~{preview.shape}")
    print(preview.head(3))

['Price - Formulas', 'Price', 'ROE - Formulas', 'ROE', 'EVtoEBITDA - Formulas', 'EVtoEBITDA', 'MarketCap - Formulas', 'MarketCap', 'GICSClassification - Formulas', 'GICS', 'SharesOutstanding - Formulas', 'SharesOutstanding']

=== Price - Formulas ===  shape~(3, 504)
        DATE  CTAS  RTX  WEC  MAA  AES  FAST  ED  EQIX  LMT  ...  WBD  KVUE  \
0 2026-06-30   NaN  NaN  NaN  NaN  NaN   NaN NaN   NaN  NaN  ...  NaN   NaN   
1 2026-05-31   NaN  NaN  NaN  NaN  NaN   NaN NaN   NaN  NaN  ...  NaN   NaN   
2 2026-04-30   NaN  NaN  NaN  NaN  NaN   NaN NaN   NaN  NaN  ...  NaN   NaN   

   COO  GEV  SOLV  SNDK   Q  CCL  FDXF  DD  
0  NaN  NaN   NaN   NaN NaN  NaN   NaN NaN  
1  NaN  NaN   NaN   NaN NaN  NaN   NaN NaN  
2  NaN  NaN   NaN   NaN NaN  NaN   NaN NaN  

[3 rows x 504 columns]

=== Price ===  shape~(3, 504)
    DATE    CTAS     RTX     WEC     MAA    AES   FAST      ED     EQIX  \
0  46203  170.08  189.73  116.77  138.94  14.66  48.03  110.63  1042.39   
1  46173  171.26  179.66  111.0

## 2. Helpers — consistent tickers & dates

Two gotchas to standardize before anything else:

- **Dates**: the `- Formulas` sheets have their `DATE` column formatted as
  real dates, but the pasted-values sheets sometimes store the *same*
  underlying dates as raw Excel serial numbers (e.g. `46203`) with no date
  number format. If that column isn't coerced explicitly, pandas will treat
  it as an integer and every later date-based join silently breaks.
  `excel_serial_to_date` below handles both cases.
- **Tickers**: uppercase + strip whitespace, so `"ctas "` and `"CTAS"` don't
  become two different tickers after a merge.

## 3. Load each field's sheet

`FIELD_SHEETS` and `SECTOR_SHEET` are set in section 0 — edit them there if
a name doesn't match what you saw printed in section 1 (in particular,
confirm the debt/equity sheet name, or delete that line if you're not using
it). Anything ending in `- Formulas` is skipped on purpose.

In [45]:
fields = {}
for field, sheet in FIELD_SHEETS.items():
    if sheet not in xls.sheet_names:
        print(f"skipping '{field}': sheet '{sheet}' not found in workbook")
        continue
    fields[field] = load_field_long(DATA_PATH, sheet, field)
    print(f"{field:<20} <- '{sheet}'  rows={len(fields[field]):>6}  "
          f"tickers={fields[field]['ticker'].nunique()}  "
          f"dates={fields[field]['date'].min().date()} .. {fields[field]['date'].max().date()}")

# ticker consistency check across sheets
ticker_sets = {f: set(df["ticker"]) for f, df in fields.items()}
common_tickers = set.intersection(*ticker_sets.values())
for f, s in ticker_sets.items():
    extra = s - common_tickers
    if extra:
        print(f"WARNING: '{f}' has tickers not shared by every sheet: {sorted(extra)}")
print("common tickers:", sorted(common_tickers))

price                <- 'Price'  rows= 59035  tickers=503  dates=2016-06-30 .. 2026-06-30
roe                  <- 'ROE'  rows= 56316  tickers=495  dates=2016-06-30 .. 2026-06-30
ev_ebitda            <- 'EVtoEBITDA'  rows= 52819  tickers=460  dates=2016-06-30 .. 2026-06-30
market_cap           <- 'MarketCap'  rows= 59071  tickers=503  dates=2016-06-30 .. 2026-06-30
shares_outstanding   <- 'SharesOutstanding'  rows= 59773  tickers=503  dates=2016-06-30 .. 2026-06-30
skipping 'debt_equity': sheet 'DebtToEquity' not found in workbook
common tickers: ['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALLE', 'AMAT', 'AMCR', 'AMD', 'AME', 'AMGN', 'AMP', 'AMT', 'AMZN', 'ANET', 'AON', 'AOS', 'APA', 'APD', 'APH', 'APO', 'APP', 'APTV', 'ARE', 'ARES', 'ATO', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP', 'BA', 'BALL', 'BAX', 'BBY', 'BDX', 'BEN', 'BF.B', 'BG', 'BIIB', 'BKNG', 'BKR', 'BLDR', 'BLK', 'BMY', 'BR', 'BRK.B', '

In [46]:
sector_map = load_sector_map(DATA_PATH, SECTOR_SHEET)
print(sector_map)

    ticker                    GICS  \
0     CTAS             Industrials   
1      RTX             Industrials   
2      WEC               Utilities   
3      MAA             Real Estate   
4      AES               Utilities   
..     ...                     ...   
498   SNDK  Information Technology   
499      Q  Information Technology   
500    CCL  Consumer Discretionary   
501   FDXF             Industrials   
502     DD               Materials   

                                GICS Sub-Industry  
0                    Diversified Support Services  
1                             Aerospace & Defense  
2                              Electric Utilities  
3                  Multi-Family Residential REITs  
4    Independent Power Producers & Energy Traders  
..                                            ...  
498    Technology Hardware, Storage & Peripherals  
499           Semiconductor Materials & Equipment  
500                Hotels, Resorts & Cruise Lines  
501                   C

## 4. Align fundamentals onto price dates

Price is the backbone — it's the frequency the model actually trades on.
Each fundamental is forward-filled *as of* the price date, but only after
adding `REPORTING_LAG_DAYS` (set in section 0): a quarter that ended on
`date` wasn't public knowledge until roughly `date + lag`, so shifting
`available_date` forward before the as-of join keeps the panel from leaking
future information into past price rows (look-ahead bias). Adjust the lag
in section 0 to match how your source reports fundamentals (45 days is a
common rough estimate for U.S. 10-Q filings; use whatever your actual
filing-lag convention is).

In [47]:
panel = fields["price"][fields["price"]["ticker"].isin(common_tickers)].copy()

fundamental_fields = [f for f in fields if f != "price"]
for f in fundamental_fields:
    fund = fields[f][fields[f]["ticker"].isin(common_tickers)]
    panel = ffill_with_reporting_lag(panel, fund, f, REPORTING_LAG_DAYS)

panel = panel.merge(sector_map, on="ticker", how="left")
panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
panel.head(10)

,date,ticker,price,roe,ev_ebitda,market_cap,shares_outstanding,GICS,GICS Sub-Industry
0,2016-06-30,A,44.36,NaN,NaN,NaN,NaN,Health Care,Life Sciences Tools & Services
1,2016-07-31,A,48.11,NaN,NaN,NaN,NaN,Health Care,Life Sciences Tools & Services
2,2016-08-31,A,46.98,11.033654,15.040648,14440.156259,331.459,Health Care,Life Sciences Tools & Services
3,2016-09-30,A,47.09,11.318968,15.040648,15660.863771,331.459,Health Care,Life Sciences Tools & Services
4,2016-10-31,A,43.57,11.318968,15.040648,15293.023560,331.459,Health Care,Life Sciences Tools & Services
5,2016-11-30,A,43.98,11.318968,15.040648,15275.290159,331.459,Health Care,Life Sciences Tools & Services
6,2016-12-31,A,45.56,10.986920,14.216426,14133.454776,324.000,Health Care,Life Sciences Tools & Services
7,2017-01-31,A,48.97,10.986920,14.216426,14266.452581,324.000,Health Care,Life Sciences Tools & Services
8,2017-02-28,A,51.30,10.986920,14.216426,14658.838677,324.000,Health Care,Life Sciences Tools & Services
9,2017-03-31,A,52.87,12.155358,14.216426,15755.999259,324.000,Health Care,Life Sciences Tools & Services


## 5. Sanity checks

Missing values right after the panel's start date are expected (there's no
prior fundamental to forward-fill from yet). Missing values in the middle
or end of the series are not — that's a sign a ticker or date mismatch
slipped through.

In [48]:
print("panel shape:", panel.shape)
print("\nmissing values per column:\n", panel.isna().sum())

print("\nrows per ticker:\n", panel.groupby("ticker").size())

# show where each fundamental first becomes non-null per ticker, to eyeball
# that forward-fill + lag look reasonable against the raw source dates
for f in fundamental_fields:
    first_valid = panel.dropna(subset=[f]).groupby("ticker")["date"].min()
    print(f"\nfirst non-null '{f}' by ticker:\n{first_valid}")

panel shape: (53224, 9)

missing values per column:
 date                     0
ticker                   0
price                    0
roe                   1035
ev_ebitda             1649
market_cap             904
shares_outstanding     865
GICS                     0
GICS Sub-Industry        0
dtype: int64

rows per ticker:
 ticker
A       121
AAPL    121
ABBV    121
ABNB     67
ABT     121
       ... 
XYZ     121
YUM     121
ZBH     121
ZBRA    121
ZTS     121
Length: 453, dtype: int64

first non-null 'roe' by ticker:
ticker
A      2016-08-31
AAPL   2016-08-31
ABBV   2016-08-31
ABNB   2021-02-28
ABT    2016-08-31
          ...    
XYZ    2016-08-31
YUM    2016-08-31
ZBH    2016-08-31
ZBRA   2016-08-31
ZTS    2016-08-31
Name: date, Length: 453, dtype: datetime64[ns]

first non-null 'ev_ebitda' by ticker:
ticker
A      2016-08-31
AAPL   2016-08-31
ABBV   2016-08-31
ABNB   2022-02-28
ABT    2016-08-31
          ...    
XYZ    2018-02-28
YUM    2016-08-31
ZBH    2016-08-31
ZBRA   2016-08

## Factor Implementation

### Value, Quality, Low Volatility

All four factors are defined so that **higher = better** — this matters
once these get z-scored and combined into a composite score later, since a
mismatched sign would have that factor pulling the composite the wrong way.

- **Value (EV/EBITDA)**: a *lower* EV/EBITDA means a cheaper stock, so the
  raw field gets negated. Negation (rather than `1/x`) is deliberate — a
  reciprocal blows up for names with EV/EBITDA near zero, which negation
  doesn't.
- **Quality (ROE)**: higher ROE is already "better," so no inversion.
- **Low volatility**: computed from monthly *returns*, not price directly —
  `price.pct_change()` per ticker, then a trailing 12-month rolling std dev
  of those returns, then negated (lower realized vol → higher factor
  value). Same as momentum, both the return calculation and the rolling
  window are grouped by ticker so one company's prices/returns never bleed
  into another's.

In [50]:
panel["mom_12_1"] = momentum_12_1(panel)
panel["value"] = value_factor(panel)
panel["quality"] = quality_factor(panel)
panel["low_vol"] = low_vol_factor(panel)

# spot-check one ticker by hand: mom_12_1 at row t should equal
# price at t-1 divided by price at t-13, minus 1
check = panel[panel["ticker"] == "AAPL"].reset_index(drop=True)
check["manual_check"] = check["price"].shift(1) / check["price"].shift(13) - 1
print("\nmatches:", check["mom_12_1"].equals(check["manual_check"]))

# spot-check low_vol by hand for one ticker: manually compute monthly
# returns and a trailing 12-month std dev, and confirm it matches
check = panel[panel["ticker"] == "AAPL"].reset_index(drop=True)
manual_returns = check["price"].pct_change()
check["manual_low_vol"] = -1 * manual_returns.rolling(12).std()
print("\nlow_vol matches manual calc:", check["low_vol"].equals(check["manual_low_vol"]))




matches: True

low_vol matches manual calc: True


## 6. Market-cap floor filter

Applied **per date**, not once at the start — a stock can cross $500M in
either direction over a 10-year window, so this has to be a per-row
eligibility flag rather than a one-time drop of tickers that happened to be
small at t=0 (which would also incorrectly keep a stock that later shrank
below the floor). Rows where `market_cap` is still `NaN` (before a ticker's
first reported fundamental) come out as `False` too — not because they're
known to be small-cap, but because eligibility can't be confirmed yet. Both
cases mean "exclude from that date's cross-section," which is the only
thing this flag needs to express for now.

In [51]:
panel["above_cap_floor"] = panel["market_cap"] >= MARKET_CAP_FLOOR

missing_cap = panel["market_cap"].isna().sum()
below_floor = (panel["market_cap"] < MARKET_CAP_FLOOR).sum()
print(f"excluded rows — missing market cap: {missing_cap}, below ${MARKET_CAP_FLOOR}M floor: {below_floor}")

# eligible universe size over time — should track close to the full
# universe, dipping only for genuinely small names
coverage = pd.DataFrame({
    "eligible": panel.groupby("date")["above_cap_floor"].sum(),
    "total": panel.groupby("date")["ticker"].nunique(),
})
coverage["pct_eligible"] = (coverage["eligible"] / coverage["total"]).round(3)
print(coverage.tail(10))

print("\ntickers ever excluded by the cap floor (missing data or sub-floor):")
print(sorted(panel.loc[~panel["above_cap_floor"], "ticker"].unique()))

excluded rows — missing market cap: 904, below $500M floor: 0
            eligible  total  pct_eligible
date                                     
2025-09-30       452    452         1.000
2025-10-31       452    453         0.998
2025-11-30       452    453         0.998
2025-12-31       453    453         1.000
2026-01-31       453    453         1.000
2026-02-28       453    453         1.000
2026-03-31       453    453         1.000
2026-04-30       453    453         1.000
2026-05-31       453    453         1.000
2026-06-30       453    453         1.000

tickers ever excluded by the cap floor (missing data or sub-floor):
['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALLE', 'AMAT', 'AMD', 'AME', 'AMGN', 'AMP', 'AMT', 'AMZN', 'ANET', 'AON', 'AOS', 'APA', 'APD', 'APH', 'APO', 'APP', 'APTV', 'ARE', 'ARES', 'ATO', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP', 'BA', 'BALL', 'BAX', 'BBY', 'BDX', 'BEN', 

## 7. Cross-sectional z-scoring, within sector, per date

A raw factor value is meaningless on its own — an EV/EBITDA of 15 doesn't
tell you if a stock is cheap or expensive without something to compare it
to. Z-scoring converts each raw value into "how many standard deviations
above/below average, *within its own comparison group*, on this date."

Two design choices worth calling out:

- **Within sector, not across the whole universe.** Utilities and tech
  stocks trade at structurally different EV/EBITDA multiples and carry
  different volatility, for reasons that have nothing to do with which one
  is the better investment. Z-scoring within `GICS` sector means a utility
  is only compared against other utilities, so the score reflects "cheap
  *for a utility*" rather than "cheap relative to Nvidia" — sector-neutral,
  matching the reference doc.
- **Only cap-floor-eligible stocks set the mean/std.** A sub-floor or
  data-missing stock (`above_cap_floor == False`) shouldn't pull the
  sector's average around, since it wouldn't be investable anyway — it's
  excluded from the population the z-score is computed against, not just
  excluded from the final portfolio.

Both a missing factor value and an ineligible stock come back as `NaN`
here, never `0` — a `0` z-score means "exactly average," which is a false
signal for a stock that has no data at all. That's what "exclude, don't
zero-fill" (reference step 7) means in practice.

In [52]:
FACTOR_COLUMNS = ["mom_12_1", "value", "quality", "low_vol"]

for col in FACTOR_COLUMNS:
    panel[f"z_{col}"] = zscore_within_sector(panel, col)

z_cols = [f"z_{c}" for c in FACTOR_COLUMNS]
print(panel[["date", "ticker", "GICS", "above_cap_floor"] + FACTOR_COLUMNS + z_cols].tail(10))

            date ticker         GICS  above_cap_floor  mom_12_1      value  \
53214 2025-09-30    ZTS  Health Care             True -0.147637 -20.236406   
53215 2025-10-31    ZTS  Health Care             True -0.251100 -20.236406   
53216 2025-11-30    ZTS  Health Care             True -0.194037 -20.236406   
53217 2025-12-31    ZTS  Health Care             True -0.268588 -20.236406   
53218 2026-01-31    ZTS  Health Care             True -0.227767 -20.236406   
53219 2026-02-28    ZTS  Health Care             True -0.269631 -15.323291   
53220 2026-03-31    ZTS  Health Care             True -0.216097 -15.323291   
53221 2026-04-30    ZTS  Health Care             True -0.282053 -15.323291   
53222 2026-05-31    ZTS  Health Care             True -0.264898 -15.323291   
53223 2026-06-30    ZTS  Health Care             True -0.539287 -15.323291   

         quality   low_vol  z_mom_12_1   z_value  z_quality  z_low_vol  
53214  52.539474 -0.058526   -0.419776  0.002615  -0.072665   0.7873

In [53]:
# sanity check: for one (date, sector) group, the eligible members' z-scores
# should have mean ~0 and std ~1 by construction
sample_date = panel["date"].max()
sample_sector = panel.loc[panel["date"] == sample_date, "GICS"].mode().iloc[0]
sample = panel[
    (panel["date"] == sample_date)
    & (panel["GICS"] == sample_sector)
    & panel["above_cap_floor"]
]
print(f"{sample_sector} on {sample_date.date()}, n={len(sample)}")
print(sample[["ticker", "mom_12_1", "z_mom_12_1"]])
print("\nz_mom_12_1 mean (should be ~0):", sample["z_mom_12_1"].mean())
print("z_mom_12_1 std (should be ~1):", sample["z_mom_12_1"].std())

Industrials on 2026-06-30, n=79
      ticker  mom_12_1  z_mom_12_1
1155     ADP -0.318527   -1.057549
2244    ALLE -0.088507   -0.626764
2692     AME  0.263567    0.032607
3539     AOS -0.118022   -0.682040
5175    AXON -0.401994   -1.213867
...      ...       ...         ...
49742   VRSK -0.442955   -1.290581
49838    VRT  1.925137    3.144424
50561    WAB  0.290826    0.083659
51408     WM -0.122463   -0.690358
52618    XYL -0.130911   -0.706178

[79 rows x 3 columns]

z_mom_12_1 mean (should be ~0): -1.1242764806330699e-17
z_mom_12_1 std (should be ~1): 0.9999999999999998


## 8. Composite score (equal-weight)

Average the four z-scores per stock per date. `DataFrame.mean(axis=1)`
skips `NaN`s by default (`skipna=True`), so a stock missing one factor
still gets a composite from the other three instead of being penalized
with an implicit zero for the missing one — consistent with "exclude, not
zero-fill." `n_factors_available` is tracked alongside so you can see —
and later decide whether to filter on — how much of each composite score
actually rests on real data versus a single available factor.

In [54]:
panel["composite_score"] = panel[z_cols].mean(axis=1, skipna=True)
panel["n_factors_available"] = panel[z_cols].notna().sum(axis=1)

print(panel[["date", "ticker"] + z_cols + ["composite_score", "n_factors_available"]].tail(10))

print("\ndistribution of n_factors_available:")
print(panel["n_factors_available"].value_counts().sort_index())

print("\nrows with a composite score but 0 factors available (should be none):",
      (panel["composite_score"].notna() & (panel["n_factors_available"] == 0)).sum())

            date ticker  z_mom_12_1   z_value  z_quality  z_low_vol  \
53214 2025-09-30    ZTS   -0.419776  0.002615  -0.072665   0.787366   
53215 2025-10-31    ZTS   -0.796223  0.002615  -0.072665   1.029993   
53216 2025-11-30    ZTS   -0.861878  0.000454  -0.082274   0.899046   
53217 2025-12-31    ZTS   -1.333378 -0.005542  -0.082294   0.946473   
53218 2026-01-31    ZTS   -1.350287 -0.005542  -0.082294   0.896932   
53219 2026-02-28    ZTS   -1.332751  0.254450  -0.142952   0.818859   
53220 2026-03-31    ZTS   -1.300893  0.237027  -0.142946   0.886595   
53221 2026-04-30    ZTS   -1.326595  0.237027  -0.142946   0.852281   
53222 2026-05-31    ZTS   -1.247089  0.230596  -0.141943  -0.631326   
53223 2026-06-30    ZTS   -1.982691  0.227045  -0.141941  -0.409632   

       composite_score  n_factors_available  
53214         0.074385                    4  
53215         0.040930                    4  
53216        -0.011163                    4  
53217        -0.118685            

## 9. IC-weighted composite (rolling)

Equal-weight treats all four factors as equally useful, which is a
reasonable starting assumption but not something we actually know. IC
(Information Coefficient) weighting instead measures, historically, how
well each factor's z-score actually lined up with what happened next, and
leans more on the factors that did.

This needs three new pieces, in order:

1. **A forward return** — the thing each factor is trying to predict.
   `price[t+1] / price[t] - 1`, per ticker.
2. **A per-date IC** — the cross-sectional (Spearman rank) correlation
   between a factor's z-score at date `t` and the forward return at `t`,
   across every stock that has both. One number per factor per date: "how
   predictive was this factor, for the move that just happened."
3. **A rolling, *lagged* average of IC** — this is the step that actually
   prevents lookahead. `IC_t` requires `price[t+1]`, which isn't known yet
   at the moment a portfolio is being formed at date `t`. So the weight
   used *at* `t` has to come from a rolling average of `IC_{t-1}` back
   through `IC_{t-window}` — never `IC_t` itself. Skipping the lag would
   mean using next month's outcome to decide this month's weights, which
   is exactly the kind of leak the reporting lag in section 4 was built to
   avoid, just showing up in a different part of the pipeline.

Negative rolling IC gets clipped to zero rather than flipped to a negative
weight — a factor with no recent predictive power gets excluded from that
date's blend, not inverted. Inverting a factor based on a noisy trailing
correlation estimate is a good way to overfit to recent noise.

In [55]:
panel["fwd_return"] = compute_forward_return(panel)

# every ticker's *last* row should be NaN — there's no "next month" price
# to look ahead to yet, which is the mirror image of momentum's warm-up NaNs
print(panel.groupby("ticker").tail(1)["fwd_return"].isna().all())

True


In [56]:
ic_df = pd.DataFrame({col: compute_ic_series(panel, col) for col in z_cols})
print(ic_df.describe())

# rolling mean IC, then shift(1) so the value attached to date t only ever
# reflects IC measured at t-1 and earlier — never t itself
rolling_ic_df = ic_df.rolling(IC_WINDOW).mean().shift(1)
print("\nrolling (lagged) IC, most recent dates:")
print(rolling_ic_df.tail())

       z_mom_12_1     z_value   z_quality   z_low_vol
count  107.000000  118.000000  118.000000  108.000000
mean     0.011213    0.001428    0.004965   -0.020867
std      0.152768    0.127359    0.073152    0.149459
min     -0.419110   -0.297449   -0.199020   -0.336425
25%     -0.107100   -0.076213   -0.047260   -0.128891
50%      0.028047    0.000895    0.007915   -0.007043
75%      0.118989    0.084586    0.049882    0.083798
max      0.330621    0.308996    0.180919    0.301437

rolling (lagged) IC, most recent dates:
            z_mom_12_1   z_value  z_quality  z_low_vol
2026-02-28   -0.014628  0.047930  -0.014523  -0.076447
2026-03-31    0.013692  0.044570  -0.016616  -0.095198
2026-04-30    0.024419  0.052705  -0.015144  -0.096784
2026-05-31    0.031300  0.063867  -0.019966  -0.094175
2026-06-30    0.046220  0.058134  -0.006572  -0.070651


### Turning rolling IC into weights, and combining

`compute_ic_weights` clips negative IC to zero and renormalizes the
positive values to sum to 1 across the four factors, per date. Two
situations both fall back to flat 25%-each weighting: the first ~`IC_WINDOW`
months of history (rolling IC is still `NaN` — not enough trailing data
yet), and any date where every factor's rolling IC happens to be zero or
negative. Both are "we don't have a reliable signal to overweight anything
with," and equal-weight is the reasonable default for that case — which
means the IC-weighted composite is *identical* to the equal-weight one for
that initial warm-up stretch, by construction, not by coincidence.

Combining the weights with each stock's z-scores has one gotcha worth
flagging explicitly: **zeroing out a weight doesn't stop a missing
z-score from poisoning the sum**, because `NaN * 0` is still `NaN`, not
`0` (that's IEEE-754 floating point, not a pandas quirk). So the missing
factor value itself has to be replaced with `0` too, *in addition to*
zeroing its weight — the zeroed weight is what keeps that `0` from
actually contributing anything once divided back out by the (correctly
shrunk) sum of weights.

**A second, sneakier gotcha, found by actually running this against real
data (see section 11's holding-period count discrepancy, and the fix
applied to compute_ic_weights above): a NaN in *one* factor's rolling IC
doesn't necessarily trigger the whole-row fallback above.** Momentum and
low-vol both need ~13 months of price history before they're even
defined, so their raw IC starts later than value and quality's. During
that gap, value/quality alone can push a row's weight_sum positive, so
the row-level fallback never fires — leaving momentum/low-vol's
individual weights as NaN instead of 0, which then poisons any stock's
composite score entirely (a valid z-score times a NaN weight is still
NaN). `.fillna(0)` before clipping fixes this: a factor with no rolling
IC yet is treated as "currently showing no positive predictive power,"
which correctly excludes it from that date's blend without leaking a NaN
into the row.

In [57]:
weights_df = compute_ic_weights(rolling_ic_df)
print(weights_df.tail())
print("\nweights sum to 1 each date:", np.allclose(weights_df.sum(axis=1), 1.0))

weight_cols = []
for col in z_cols:
    wcol = f"w_{col}"
    panel[wcol] = panel["date"].map(weights_df[col])
    weight_cols.append(wcol)

panel["composite_ic_weighted"] = combine_weighted(panel, z_cols, weight_cols)
print(panel[["date", "ticker", "composite_score", "composite_ic_weighted"]].tail(10))

            z_mom_12_1   z_value  z_quality  z_low_vol
2026-02-28    0.000000  1.000000        0.0        0.0
2026-03-31    0.235006  0.764994        0.0        0.0
2026-04-30    0.316618  0.683382        0.0        0.0
2026-05-31    0.328895  0.671105        0.0        0.0
2026-06-30    0.442914  0.557086        0.0        0.0

weights sum to 1 each date: True
            date ticker  composite_score  composite_ic_weighted
53214 2025-09-30    ZTS         0.074385              -0.237017
53215 2025-10-31    ZTS         0.040930               0.002615
53216 2025-11-30    ZTS        -0.011163               0.000454
53217 2025-12-31    ZTS        -0.118685              -0.005542
53218 2026-01-31    ZTS        -0.135298              -0.005542
53219 2026-02-28    ZTS        -0.100599               0.254450
53220 2026-03-31    ZTS        -0.080055              -0.124394
53221 2026-04-30    ZTS        -0.095058              -0.258044
53222 2026-05-31    ZTS        -0.447441              -0.255

In [58]:
# --- sanity checks ---

# during the warm-up window (before IC_WINDOW months of IC history exist),
# the two composites should be exactly equal, by construction
warmup_dates = ic_df.index[:IC_WINDOW]
warmup_rows = panel[panel["date"].isin(warmup_dates)]
print("composites match during warm-up:",
      np.allclose(warmup_rows["composite_score"], warmup_rows["composite_ic_weighted"], equal_nan=True))

# after warm-up, they should generally differ (IC-weighting is doing something)
# but stay reasonably correlated (both are still blends of the same 4 factors)
post_warmup = panel[~panel["date"].isin(warmup_dates)].dropna(subset=["composite_score", "composite_ic_weighted"])
print("correlation between the two composites post-warmup:",
      post_warmup["composite_score"].corr(post_warmup["composite_ic_weighted"]))

# factor efficacy summary — mean raw (unlagged) IC and % of dates with a
# positive IC per factor, i.e. "how predictive was each factor overall"
print("\nfactor efficacy (mean IC, % positive months):")
print(pd.DataFrame({
    "mean_ic": ic_df.mean(),
    "pct_positive": (ic_df > 0).mean(),
}))

composites match during warm-up: True
correlation between the two composites post-warmup: 0.6095408352889832

factor efficacy (mean IC, % positive months):
             mean_ic  pct_positive
z_mom_12_1  0.011213      0.520661
z_value     0.001428      0.487603
z_quality   0.004965      0.528926
z_low_vol  -0.020867      0.396694


### IC quality: Information Ratio and significance

Mean IC alone conflates "how strong" with "how consistent." `ic_ir` (IC
Information Ratio) is `mean(IC) / std(IC)` — dividing by the standard
deviation penalizes a factor whose IC swings between strongly positive and
strongly negative month to month, even if the average happens to look
fine. A factor with a small but *stable* mean IC (high IR) is more
trustworthy than one with a larger but erratic mean IC (low IR).

`t_stat` reuses that same ratio, scaled by `sqrt(n)` (`t = IC_IR × √n`, the
standard formula — it falls out of the fact that the standard error of a
mean is `std / √n`, so `mean / (std/√n)` simplifies to `(mean/std) × √n`).
This asks whether `mean_ic` is distinguishable from what pure noise would
produce given only `n` ≈ 121 monthly observations, not just whether it
happens to be positive. As a rough rule of thumb, `|t_stat|` needs to be
around 2 or higher before "this factor has real predictive power" is a
safer read than "this factor's IC randomly averaged above zero over this
particular sample."

In [59]:
n_obs = len(ic_df)
ic_ir = ic_df.mean() / ic_df.std()
ic_tstat = ic_ir * np.sqrt(n_obs)

ic_quality = pd.DataFrame({
    "mean_ic": ic_df.mean(),
    "ic_std": ic_df.std(),
    "ic_ir": ic_ir,
    "t_stat": ic_tstat,
})
print(f"n_obs = {n_obs}\n")
print(ic_quality)

n_obs = 121

             mean_ic    ic_std     ic_ir    t_stat
z_mom_12_1  0.011213  0.152768  0.073398  0.807382
z_value     0.001428  0.127359  0.011214  0.123353
z_quality   0.004965  0.073152  0.067867  0.746542
z_low_vol  -0.020867  0.149459 -0.139617 -1.535790


## 10. Kalman-filtered composite

Sections 8–9 built two ways to *weight* the four factors. This section
builds a third, following the reference doc's actual wording — "smooth
factor return/covariance estimates before feeding into optimization" —
which needs two ingredients we don't have yet, plus an optimizer:

1. **Factor returns.** Not correlation (IC) — an actual return series per
   factor, in the same units as a stock return. Built via a Fama-MacBeth
   cross-sectional regression: each month, regress every eligible stock's
   forward return on its z-score for one factor
   (`forward_return_i = alpha + beta * z_i + error_i`); the slope `beta`
   is "the return earned per unit of exposure to this factor that month."
   Four factors → four return series, structurally parallel to `ic_df`
   from section 9, but a regression slope instead of a rank correlation.
2. **A covariance matrix between the four factor-return series.** This
   needs a trailing window of history (covariance isn't defined from a
   single time point), so it starts from a rolling covariance, which then
   gets Kalman-smoothed the same way the mean does.
3. **Optimization.** With a (smoothed) expected return per factor and a
   (smoothed) covariance between them, solve for the combination of the
   four factors that maximizes Sharpe ratio — `w ∝ Σ⁻¹μ`, the classical
   tangency-portfolio formula from Markowitz mean-variance optimization,
   just applied to 4 factors instead of hundreds of stocks. This is the
   part that's genuinely new relative to section 9 — IC-weighting only
   ever looked at one factor at a time, this looks at how all four move
   *together* and can down-weight a factor for being redundant with
   another, not just for being individually weak.

Both `μ` (mean vector) and `Σ` (covariance matrix) get the same one-period
lag as before — both are built from `fwd_return`, which needs `price[t+1]`,
so the estimate used to weight date `t` still can't include date `t`
itself. And exactly as with IC-weighting, the optimizer's raw output is
clipped at zero and renormalized (long-only across factors — a factor the
optimizer wants to short gets excluded, not inverted) and falls back to
equal weight during warm-up or if `Σ` is singular/unusable.

One honest limitation worth stating outright: smoothing each entry of the
covariance matrix independently (rather than filtering the whole matrix
jointly) is a simplification — it isn't mathematically guaranteed to
produce a valid (positive semi-definite) covariance matrix every period.
A full multivariate Kalman filter would guarantee that, at the cost of
real complexity; the code below instead detects and gracefully falls back
whenever a given date's matrix turns out to be unusable, which is a
reasonable approximation for a 4-factor combination but worth documenting
as a known simplification, the same way the reporting-lag approximation
and Wikipedia sector fallback are documented in the reference doc.

In [61]:
factor_returns_df = pd.DataFrame({col: compute_factor_return_series(panel, col) for col in z_cols})
print(factor_returns_df.describe())

       z_mom_12_1     z_value   z_quality   z_low_vol
count  107.000000  118.000000  118.000000  108.000000
mean     0.002957   -0.000718   -0.000092   -0.005185
std      0.013835    0.009836    0.004252    0.015883
min     -0.042137   -0.024481   -0.014864   -0.053409
25%     -0.004312   -0.006819   -0.002378   -0.015502
50%      0.003121   -0.000794    0.000012   -0.003804
75%      0.009404    0.005698    0.002502    0.005271
max      0.054896    0.024815    0.013113    0.024927


In [62]:
# --- smooth the mean vector: one Kalman filter per factor, reusing the
# exact same function already verified by weighting.py's self-test ---
smoothed_mean_df = pd.DataFrame({
    col: kalman_filter_series(factor_returns_df[col], KALMAN_Q_TO_R_RATIO) for col in z_cols
})
smoothed_mean_lagged = smoothed_mean_df.shift(1)

# --- smooth the covariance matrix: rolling covariance first (needs a
# window of history to even be defined), then Kalman-filter each unique
# entry independently across time ---
raw_cov_entries = rolling_covariance_entries(factor_returns_df, IC_WINDOW)
smoothed_cov_entries = pd.DataFrame({
    pair: kalman_filter_series(raw_cov_entries[pair], KALMAN_Q_TO_R_RATIO)
    for pair in raw_cov_entries.columns
})
smoothed_cov_entries_lagged = smoothed_cov_entries.shift(1)

print(smoothed_mean_lagged.tail())
print("\nsmoothed covariance entries, most recent dates:")
print(smoothed_cov_entries_lagged.tail())

            z_mom_12_1   z_value  z_quality  z_low_vol
2026-02-28    0.007069  0.007673   0.000411  -0.008547
2026-03-31    0.006121  0.007046  -0.000917  -0.006499
2026-04-30    0.012646  0.006030   0.001889  -0.010419
2026-05-31    0.011703  0.001816   0.001404  -0.012646
2026-06-30    0.011139  0.002060   0.002984  -0.011040

smoothed covariance entries, most recent dates:
           z_mom_12_1                                 z_value            \
           z_mom_12_1   z_value z_quality z_low_vol   z_value z_quality   
2026-02-28   0.000210 -0.000074  0.000041 -0.000145  0.000094 -0.000012   
2026-03-31   0.000190 -0.000065  0.000038 -0.000123  0.000099 -0.000011   
2026-04-30   0.000189 -0.000054  0.000043 -0.000114  0.000094 -0.000009   
2026-05-31   0.000188 -0.000045  0.000046 -0.000104  0.000093 -0.000006   
2026-06-30   0.000185 -0.000038  0.000048 -0.000099  0.000093 -0.000004   

                     z_quality           z_low_vol  
           z_low_vol z_quality z_low_vol z

In [63]:
kalman_opt_weights = {}
for date in factor_returns_df.index:
    mean_vec = smoothed_mean_lagged.loc[date, z_cols].to_numpy()
    cov_mat = build_cov_matrix(smoothed_cov_entries_lagged.loc[date], z_cols)
    cov_mat = shrink_covariance(cov_mat, COV_SHRINKAGE)
    kalman_opt_weights[date] = mean_variance_weights(mean_vec, cov_mat)

kalman_weights_df = pd.DataFrame(kalman_opt_weights, index=z_cols).T
print(kalman_weights_df.tail(10))
print("\nweights sum to 1 each date:", np.allclose(kalman_weights_df.sum(axis=1), 1.0))
print("dates using the equal-weight fallback:", (kalman_weights_df == 1 / len(z_cols)).all(axis=1).sum())

shrink_covariance endpoints verified: shrinkage=0 unchanged, shrinkage=1 fully diagonal
            z_mom_12_1   z_value  z_quality  z_low_vol
2025-09-30    0.356416  0.643584    0.00000        0.0
2025-10-31    0.315460  0.684540    0.00000        0.0
2025-11-30    0.369555  0.630445    0.00000        0.0
2025-12-31    0.358103  0.641897    0.00000        0.0
2026-01-31    0.281588  0.718412    0.00000        0.0
2026-02-28    0.282978  0.717022    0.00000        0.0
2026-03-31    0.342380  0.657620    0.00000        0.0
2026-04-30    0.380531  0.619469    0.00000        0.0
2026-05-31    0.453007  0.546993    0.00000        0.0
2026-06-30    0.388518  0.565402    0.04608        0.0

weights sum to 1 each date: True
dates using the equal-weight fallback: 31


In [64]:
kalman_weight_cols = []
for col in z_cols:
    wcol = f"wk_{col}"
    panel[wcol] = panel["date"].map(kalman_weights_df[col])
    kalman_weight_cols.append(wcol)

# reusing combine_weighted unchanged from section 9 — it only ever needed
# "a z-score column and a matching weight column, same order," which is
# just as true for these optimizer-derived weights as for the IC ones
panel["composite_kalman_weighted"] = combine_weighted(panel, z_cols, kalman_weight_cols)

print(panel[["date", "ticker", "composite_score", "composite_ic_weighted", "composite_kalman_weighted"]].tail(10))

            date ticker  composite_score  composite_ic_weighted  \
53214 2025-09-30    ZTS         0.074385              -0.237017   
53215 2025-10-31    ZTS         0.040930               0.002615   
53216 2025-11-30    ZTS        -0.011163               0.000454   
53217 2025-12-31    ZTS        -0.118685              -0.005542   
53218 2026-01-31    ZTS        -0.135298              -0.005542   
53219 2026-02-28    ZTS        -0.100599               0.254450   
53220 2026-03-31    ZTS        -0.080055              -0.124394   
53221 2026-04-30    ZTS        -0.095058              -0.258044   
53222 2026-05-31    ZTS        -0.447441              -0.255408   
53223 2026-06-30    ZTS        -0.576805              -0.751678   

       composite_kalman_weighted  
53214                  -0.147932  
53215                  -0.249387  
53216                  -0.318225  
53217                  -0.481044  
53218                  -0.384207  
53219                  -0.194693  
53220            

In [65]:
# --- sanity checks ---

three = panel[["composite_score", "composite_ic_weighted", "composite_kalman_weighted"]].dropna()
print("correlation matrix across all three composites:")
print(three.corr())

print(f"\ndates using the equal-weight fallback (warm-up or unusable Σ): "
      f"{(kalman_weights_df == 1 / len(z_cols)).all(axis=1).sum()} / {len(kalman_weights_df)}")

# weight path over the last year — should move more gradually than the
# rolling-IC weights did, since a Kalman filter's gain shrinks as it
# accumulates history, rather than a fixed window jumping wholesale in and
# out of the calculation each month
print("\nKalman-optimized weight path, last 12 months:")
print(kalman_weights_df.tail(12).round(3))

correlation matrix across all three composites:
                           composite_score  composite_ic_weighted  \
composite_score                   1.000000               0.642449   
composite_ic_weighted             0.642449               1.000000   
composite_kalman_weighted         0.673480               0.654370   

                           composite_kalman_weighted  
composite_score                              0.67348  
composite_ic_weighted                        0.65437  
composite_kalman_weighted                    1.00000  

dates using the equal-weight fallback (warm-up or unusable Σ): 31 / 121

Kalman-optimized weight path, last 12 months:
            z_mom_12_1  z_value  z_quality  z_low_vol
2025-07-31       0.250    0.250      0.250       0.25
2025-08-31       0.000    1.000      0.000       0.00
2025-09-30       0.356    0.644      0.000       0.00
2025-10-31       0.315    0.685      0.000       0.00
2025-11-30       0.370    0.630      0.000       0.00
2025-12-31 

## 11. Portfolio construction

Turns each composite score into an actual, tradeable portfolio: at every
quarterly rebalance date, rank the cap-floor-eligible universe by that
date's composite score, buy the top quintile, weight every holding
equally within it, and hold with no interim trading until the next
rebalance date. Built for all three composites from sections 8-10, so the
question from last turn -- equal-weight vs. IC-weighted vs.
Kalman-weighted -- finally gets an answer grounded in realized returns
instead of just correlation with next-month returns.

**Rebalance dates**: calendar quarter-ends (`REBALANCE_MONTHS`, section
0) subset from the existing monthly panel -- no new date logic needed,
since the panel is already month-end price data and a quarter-end is just
every third one of those.

**Why total price change, not compounded monthly returns, for the
holding-period return.** Between two rebalance dates there's no interim
trading -- the portfolio is bought once and held. For a stock held with
no trades in between, the total return over the period is exactly
`price[end] / price[start] - 1`, regardless of how it moved in between.
Compounding monthly legs would give the identical number for a static
holding; going straight to the endpoints is just simpler.

A stock that drops out of the panel before the next rebalance date (a data
gap -- shouldn't happen in this survivorship-biased, current-constituents
dataset, but the code doesn't assume that) is excluded from that period's
return calculation rather than counted as a 0% or -100% return -- same
"exclude, don't invent a number" principle used everywhere else in this
notebook.

In [66]:
rebalance_dates = sorted(panel.loc[panel["date"].dt.month.isin(REBALANCE_MONTHS), "date"].unique())
print(f"{len(rebalance_dates)} quarterly rebalance dates: {pd.Timestamp(rebalance_dates[0]).date()} .. {pd.Timestamp(rebalance_dates[-1]).date()}")

41 quarterly rebalance dates: 2016-06-30 .. 2026-06-30


In [67]:
SCORE_COLS = ["composite_score", "composite_ic_weighted", "composite_kalman_weighted"]
portfolios = {col: compute_portfolio_returns(panel, col, rebalance_dates) for col in SCORE_COLS}

for name, df in portfolios.items():
    print(f"\n{name}: {len(df)} holding periods, avg n_holdings={df['n_holdings'].mean():.1f}")
    print(df.tail())


composite_score: 39 holding periods, avg n_holdings=88.0
            n_holdings  n_priced  portfolio_return
date                                              
2025-06-30          90        90          0.071283
2025-09-30          90        90          0.044412
2025-12-31          91        91         -0.009632
2026-03-31          91        91          0.049742
2026-06-30          91        91          0.088526

composite_ic_weighted: 39 holding periods, avg n_holdings=87.9
            n_holdings  n_priced  portfolio_return
date                                              
2025-06-30          90        90          0.029714
2025-09-30          90        90          0.081875
2025-12-31          91        91          0.059189
2026-03-31          91        91          0.069479
2026-06-30          91        91          0.134748

composite_kalman_weighted: 39 holding periods, avg n_holdings=87.8
            n_holdings  n_priced  portfolio_return
date                                         

In [68]:
# --- sanity checks ---

# spot-check one holding period by hand for one ticker
check_date = rebalance_dates[-2]
check_next = rebalance_dates[-1]
check_holdings = quintile_top_holdings(panel, "composite_score", check_date)
check_ticker = check_holdings[0]
manual_start = panel.loc[(panel["date"] == check_date) & (panel["ticker"] == check_ticker), "price"].iloc[0]
manual_end = panel.loc[(panel["date"] == check_next) & (panel["ticker"] == check_ticker), "price"].iloc[0]
manual_return = manual_end / manual_start - 1
computed_return = (
    panel[(panel["date"] == check_next) & (panel["ticker"].isin(check_holdings))]
    .set_index("ticker")["price"]
    / panel[(panel["date"] == check_date) & (panel["ticker"].isin(check_holdings))].set_index("ticker")["price"]
    - 1
)
print(f"{check_ticker} holding-period return, manual: {manual_return:.6f}, "
      f"from computed_return: {computed_return.loc[check_ticker]:.6f}, "
      f"match: {np.isclose(manual_return, computed_return.loc[check_ticker])}")

# quintile size should be roughly 1/5 of that date's eligible universe
eligible_count = panel[(panel["date"] == check_date) & panel["above_cap_floor"]]["ticker"].nunique()
print(f"\neligible universe on {pd.Timestamp(check_date).date()}: {eligible_count}, "
      f"top-quintile holdings: {len(check_holdings)} (expected ~{eligible_count / 5:.0f})")


def compute_turnover(panel: pd.DataFrame, score_col: str, rebalance_dates: list) -> pd.Series:
    """Fraction of the prior quarter's holdings that got replaced at each
    rebalance -- high turnover isn't a bug, but it's a direct input to the
    transaction-cost drag the reference doc's backtest section (§7) calls
    for, so worth tracking now that holdings exist to compare."""
    turnovers = {}
    prev_holdings = None
    for date in rebalance_dates:
        holdings = set(quintile_top_holdings(panel, score_col, date))
        if prev_holdings:
            turnovers[date] = len(holdings - prev_holdings) / len(prev_holdings)
        prev_holdings = holdings
    return pd.Series(turnovers)


for name in SCORE_COLS:
    turnover = compute_turnover(panel, name, rebalance_dates)
    print(f"\n{name} -- average quarterly turnover: {turnover.mean():.1%}")

AAPL holding-period return, manual: 0.140155, from computed_return: 0.140155, match: True

eligible universe on 2026-03-31: 453, top-quintile holdings: 91 (expected ~91)

composite_score -- average quarterly turnover: 31.6%

composite_ic_weighted -- average quarterly turnover: 42.7%

composite_kalman_weighted -- average quarterly turnover: 46.2%


In [69]:
# --- compare realized performance across the three composites ---
returns_comparison = pd.DataFrame({name: df["portfolio_return"] for name, df in portfolios.items()})
cumulative_growth = (1 + returns_comparison).cumprod()

print("quarterly returns, most recent:")
print(returns_comparison.tail())

print("\ngrowth of $1 invested at the start of the backtest:")
print(cumulative_growth.tail())

print("\nmean quarterly return, and % of quarters positive:")
print(pd.DataFrame({
    "mean_quarterly_return": returns_comparison.mean(),
    "pct_positive_quarters": (returns_comparison > 0).mean(),
}))

quarterly returns, most recent:
            composite_score  composite_ic_weighted  composite_kalman_weighted
date                                                                         
2025-06-30         0.071283               0.029714                   0.073135
2025-09-30         0.044412               0.081875                   0.061699
2025-12-31        -0.009632               0.059189                   0.059156
2026-03-31         0.049742               0.069479                   0.094067
2026-06-30         0.088526               0.134748                   0.150515

growth of $1 invested at the start of the backtest:
            composite_score  composite_ic_weighted  composite_kalman_weighted
date                                                                         
2025-06-30         3.472782               2.842783                   3.419396
2025-09-30         3.627014               3.075536                   3.630369
2025-12-31         3.592081               3.257574       

## 12. Backtest mechanics & performance metrics

Turns the gross quarterly returns from section 11 into the numbers your
reference doc's sections 7-8 actually ask for: net-of-cost returns, a
real benchmark, and the standard backtest metrics (annualized return/vol,
Sharpe, max drawdown).

**No-lookahead and survivorship bias (section 7)**: no new code needed
here -- the point-in-time fundamentals lag (section 4) and the
reporting-lag/eligibility rules already threaded through every factor and
every rebalance decision *are* the no-lookahead guarantee. Survivorship
bias remains an open, documented limitation: this dataset is built from
*current* S&P 500 constituents, so it can't see companies that were
removed from the index (acquired, delisted, dropped for underperformance)
over the sample window -- the backtest below is systematically biased
toward surviving winners, and that should be stated plainly in any
writeup of these results, not left implicit.

**Transaction costs (section 7)**: `TRANSACTION_COST_BPS = 10` (section
0), applied as `cost = TRANSACTION_COST_BPS / 10000 * turnover` for each
holding period -- a direct reading of "10bps per unit of turnover
traded," not doubled for a round trip. That's a stated modeling choice,
not a universal convention -- worth adjusting if you have a firmer number
for a specific venue/broker.

**Benchmark (section 8)**: rather than pull in an external SPY series,
the benchmark here is an equal-weighted portfolio of the *entire*
cap-floor-eligible universe, no factor tilt at all -- built from data
already in this notebook, using the exact same buy-and-hold-between-
rebalances mechanics as the three factor portfolios. This is literally
the "equal-weighted S&P 500" alternative your reference doc names.

**Annualized return/vol, Sharpe, max drawdown (section 8)**: standard
definitions, computed in `performance_metrics` below -- `annualized_return`
compounds geometrically (matching how the growth-of-$1 curve in section
11 was built, not a naive x4 multiply of the average quarterly return),
`annualized_vol` scales the quarterly standard deviation by sqrt(4) (the
standard iid-returns approximation -- a real return series isn't
perfectly iid, but this is the conventional simplification), and
`RISK_FREE_RATE = 0.0` (section 0) is itself a stated simplification -- a
more rigorous Sharpe would subtract a contemporaneous T-bill yield
instead of comparing straight to zero.

In [70]:
benchmark_returns = compute_benchmark_returns(panel, rebalance_dates)
print(f"benchmark: {len(benchmark_returns)} holding periods, avg n_holdings={benchmark_returns['n_holdings'].mean():.1f}")
print(benchmark_returns.tail())

benchmark: 39 holding periods, avg n_holdings=439.4
            n_holdings  portfolio_return
date                                    
2025-06-30         451          0.077206
2025-09-30         452          0.062888
2025-12-31         452          0.016833
2026-03-31         453          0.016505
2026-06-30         453          0.111869


In [71]:
cost_rate = TRANSACTION_COST_BPS / 10000
net_returns = {}
for name in SCORE_COLS:
    turnover = compute_turnover_by_period(panel, name, rebalance_dates)
    gross = portfolios[name]["portfolio_return"]
    turnover_aligned = turnover.reindex(gross.index).fillna(0)
    net_returns[name] = gross - cost_rate * turnover_aligned

    print(f"{name}: avg turnover={turnover.mean():.1%}, "
          f"avg cost drag per period={cost_rate * turnover.mean():.4%}")

composite_score: avg turnover=31.7%, avg cost drag per period=0.0317%
composite_ic_weighted: avg turnover=43.0%, avg cost drag per period=0.0430%
composite_kalman_weighted: avg turnover=46.6%, avg cost drag per period=0.0466%


In [73]:
results = {}
for name in SCORE_COLS:
    results[f"{name}_gross"] = performance_metrics(portfolios[name]["portfolio_return"], PERIODS_PER_YEAR, RISK_FREE_RATE)
    results[f"{name}_net"] = performance_metrics(net_returns[name], PERIODS_PER_YEAR, RISK_FREE_RATE)
results["benchmark_equal_weight"] = performance_metrics(benchmark_returns["portfolio_return"], PERIODS_PER_YEAR, RISK_FREE_RATE)

results_table = pd.DataFrame(results).T
print(results_table.round(4))

# --- sanity checks ---
# net-of-cost return can never exceed gross return -- costs only subtract
for name in SCORE_COLS:
    gross = portfolios[name]["portfolio_return"]
    net = net_returns[name].reindex(gross.index)
    print(f"{name}: net <= gross every period:", (net <= gross + 1e-9).all())

print("\nall max_drawdowns <= 0:", (results_table["max_drawdown"] <= 0).all())

print("\nformatted summary:")
display_table = results_table.copy()
for col in ["annualized_return", "annualized_vol", "max_drawdown"]:
    display_table[col] = (display_table[col] * 100).round(2).astype(str) + "%"
display_table["sharpe_ratio"] = display_table["sharpe_ratio"].round(3)
print(display_table[["annualized_return", "annualized_vol", "sharpe_ratio", "max_drawdown", "n_periods"]])

                                 annualized_return  annualized_vol  \
composite_score_gross                       0.1558          0.1681   
composite_score_net                         0.1545          0.1681   
composite_ic_weighted_gross                 0.1514          0.1736   
composite_ic_weighted_net                   0.1495          0.1736   
composite_kalman_weighted_gross             0.1755          0.1796   
composite_kalman_weighted_net               0.1735          0.1796   
benchmark_equal_weight                      0.1577          0.1739   

                                 sharpe_ratio  max_drawdown  n_periods  
composite_score_gross                  0.9271       -0.2327       39.0  
composite_score_net                    0.9187       -0.2331       39.0  
composite_ic_weighted_gross            0.8723       -0.2059       39.0  
composite_ic_weighted_net              0.8612       -0.2072       39.0  
composite_kalman_weighted_gross        0.9774       -0.2327       39.0  
c

## 13. Long-short: isolating the factor premium from market beta

Every portfolio built so far, including the "benchmark," is long-only --
which means most of the ~15-18% annualized return everywhere in section
12's table is just broad market exposure (the whole eligible universe
went up over this window), not evidence that any of these factors
actually picks better stocks. A dollar-neutral long-short portfolio --
long the top quintile, short the bottom quintile, equal-weighted within
each leg -- cancels out a pure market move (if everything rises 5% that
month, the long leg's +5% and the short leg's -(-5%) roughly net to zero)
and isolates whatever return difference is actually attributable to the
factor's own ranking. This is the cleaner test of "does this factor
combination add anything," separate from "did the market go up."

Turnover (and therefore transaction cost) is summed across *both* legs
here, since both get rebalanced every quarter -- a long-short portfolio
trades roughly twice as much per unit of quintile churn as a long-only
one.

In [74]:
# spot-check: long and short legs must never overlap on a given date
_check_date = rebalance_dates[-2]
_long = set(quintile_top_holdings(panel, "composite_score", _check_date))
_short = set(quintile_bottom_holdings(panel, "composite_score", _check_date))
print("long/short legs disjoint on", pd.Timestamp(_check_date).date(), ":", _long.isdisjoint(_short))

long_short_portfolios = {name: compute_long_short_returns(panel, name, rebalance_dates) for name in SCORE_COLS}
for name, df in long_short_portfolios.items():
    print(f"\n{name}: {len(df)} periods, avg n_long={df['n_long'].mean():.1f}, avg n_short={df['n_short'].mean():.1f}")
    print(df[["long_return", "short_return", "long_short_return"]].tail())

long/short legs disjoint on 2026-03-31 : True

composite_score: 39 periods, avg n_long=88.0, avg n_short=88.3
            long_return  short_return  long_short_return
date                                                    
2025-06-30     0.071283      0.123815          -0.052532
2025-09-30     0.044412      0.083705          -0.039293
2025-12-31    -0.009632      0.037128          -0.046760
2026-03-31     0.049742     -0.023461           0.073203
2026-06-30     0.088526      0.108185          -0.019659

composite_ic_weighted: 39 periods, avg n_long=87.9, avg n_short=88.1
            long_return  short_return  long_short_return
date                                                    
2025-06-30     0.029714      0.156166          -0.126452
2025-09-30     0.081875      0.089098          -0.007223
2025-12-31     0.059189      0.015318           0.043870
2026-03-31     0.069479     -0.030428           0.099907
2026-06-30     0.134748      0.064951           0.069797

composite_kalman_weig

In [75]:
long_short_net_returns = {}
for name in SCORE_COLS:
    gross = long_short_portfolios[name]["long_short_return"]
    turnover = compute_long_short_turnover(panel, name, rebalance_dates)
    turnover_aligned = turnover.reindex(gross.index).fillna(0)
    long_short_net_returns[name] = gross - cost_rate * turnover_aligned

long_short_results = {}
for name in SCORE_COLS:
    long_short_results[f"{name}_gross"] = performance_metrics(long_short_portfolios[name]["long_short_return"], PERIODS_PER_YEAR, RISK_FREE_RATE)
    long_short_results[f"{name}_net"] = performance_metrics(long_short_net_returns[name], PERIODS_PER_YEAR, RISK_FREE_RATE)

long_short_results_table = pd.DataFrame(long_short_results).T
print(long_short_results_table.round(4))

                                 annualized_return  annualized_vol  \
composite_score_gross                      -0.0423          0.1114   
composite_score_net                        -0.0445          0.1114   
composite_ic_weighted_gross                -0.0300          0.1222   
composite_ic_weighted_net                  -0.0332          0.1222   
composite_kalman_weighted_gross            -0.0075          0.1330   
composite_kalman_weighted_net              -0.0111          0.1330   

                                 sharpe_ratio  max_drawdown  n_periods  
composite_score_gross                 -0.3797       -0.4242       39.0  
composite_score_net                   -0.3993       -0.4338       39.0  
composite_ic_weighted_gross           -0.2458       -0.4524       39.0  
composite_ic_weighted_net             -0.2719       -0.4646       39.0  
composite_kalman_weighted_gross       -0.0564       -0.4408       39.0  
composite_kalman_weighted_net         -0.0831       -0.4532       39.0 

## 14. Is the Sharpe difference between strategies real, or noise?

Section 12 ranked four strategies by Sharpe ratio using a single point
estimate each, from 39 quarterly observations -- exactly the kind of
small-sample situation where "0.995 beats 0.919" can look decisive
without actually being distinguishable from chance. This applies the same
skepticism to the final backtest comparison that section 9's IC t-stats
applied to the individual factors.

**Block bootstrap, not a plain (iid) bootstrap.** An ordinary bootstrap
resamples single quarters independently, which implicitly assumes
consecutive quarters' returns don't depend on each other -- a real
return series generally has at least some short-run autocorrelation
(momentum/mean-reversion effects, clustered volatility). Resampling
contiguous blocks of `BLOCK_LENGTH` quarters (with wraparound) instead
preserves whatever of that structure exists in the actual data.

**Paired, not independent, resampling across strategies.** Each bootstrap
draw applies the *same* resampled sequence of dates to all four return
series at once, because the strategies aren't independent of each other
-- they hold overlapping stocks and lived through the same historical
quarters. Bootstrapping each series separately would treat "the market
had a rough quarter" as unrelated noise for each strategy individually,
overstating how uncertain the *differences between them* really are.

In [76]:
def sharpe_metric(returns: pd.Series) -> float:
    return performance_metrics(returns, PERIODS_PER_YEAR, RISK_FREE_RATE)["sharpe_ratio"]


returns_for_bootstrap = {**net_returns, "benchmark_equal_weight": benchmark_returns["portfolio_return"]}
sharpe_boot = bootstrap_metric_distribution(returns_for_bootstrap, sharpe_metric, N_BOOTSTRAP, BLOCK_LENGTH)

ci_table = pd.DataFrame({
    "point_estimate": {name: sharpe_metric(pd.Series(returns_for_bootstrap[name]).dropna()) for name in returns_for_bootstrap},
    "bootstrap_median": sharpe_boot.median(),
    "ci_5": sharpe_boot.quantile(0.05),
    "ci_95": sharpe_boot.quantile(0.95),
})
print(ci_table.round(3))

                           point_estimate  bootstrap_median   ci_5  ci_95
composite_score                     0.919             0.944  0.381  1.859
composite_ic_weighted               0.861             0.885  0.352  1.686
composite_kalman_weighted           0.966             0.989  0.435  1.807
benchmark_equal_weight              0.907             0.939  0.396  1.766


In [77]:
from itertools import combinations

pairs_summary = []
for a, b in combinations(sharpe_boot.columns, 2):
    diff = sharpe_boot[a] - sharpe_boot[b]
    pairs_summary.append({
        "comparison": f"{a} minus {b}",
        "median_diff": diff.median(),
        "ci_5": diff.quantile(0.05),
        "ci_95": diff.quantile(0.95),
        "pct_a_beats_b": (diff > 0).mean(),
    })

pairs_df = pd.DataFrame(pairs_summary).set_index("comparison")
print(pairs_df.round(3))

print("\nA 90% CI on the difference that includes 0 means the two strategies'")
print("Sharpe ratios are not distinguishable from each other at that confidence")
print("level, given this sample -- the point-estimate ranking from section 12")
print("may simply reflect which strategy got luckier over these 39 quarters.")

                                                    median_diff   ci_5  ci_95  \
comparison                                                                      
composite_score minus composite_ic_weighted               0.077 -0.143  0.332   
composite_score minus composite_kalman_weighted          -0.029 -0.248  0.217   
composite_score minus benchmark_equal_weight              0.017 -0.119  0.171   
composite_ic_weighted minus composite_kalman_we...       -0.103 -0.229 -0.006   
composite_ic_weighted minus benchmark_equal_weight       -0.057 -0.299  0.185   
composite_kalman_weighted minus benchmark_equal...        0.044 -0.178  0.291   

                                                    pct_a_beats_b  
comparison                                                         
composite_score minus composite_ic_weighted                 0.718  
composite_score minus composite_kalman_weighted             0.399  
composite_score minus benchmark_equal_weight                0.576  
composite_i

## 15. Deliverable charts

Two of these are required by the reference doc's deliverables section
(equity curve comparison, factor performance breakdown); the other three
visualize findings from this session that were otherwise only visible in
printed tables -- weight concentration over time, drawdown, and the
long-short result. Every chart reuses data already computed above --
nothing new is calculated here. Each is saved to `results/` as a PNG
(matching the structure `README.md` already points to) in addition to
rendering inline.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

STRATEGY_COLORS = {
    "composite_score": "#1f77b4",
    "composite_ic_weighted": "#ff7f0e",
    "composite_kalman_weighted": "#2ca02c",
    "benchmark_equal_weight": "#7f7f7f",
}
STRATEGY_LABELS = {
    "composite_score": "Equal-weight",
    "composite_ic_weighted": "IC-weighted",
    "composite_kalman_weighted": "Kalman-weighted",
    "benchmark_equal_weight": "Benchmark (equal-weight universe)",
}

# net-of-cost returns for all four series, one dict for reuse across charts
# -- the benchmark has no factor-driven turnover to net out, so its gross
# return from section 12 is already its "net" return
all_net_returns = {**net_returns, "benchmark_equal_weight": benchmark_returns["portfolio_return"]}

In [ ]:
# --- Chart 1: equity curve comparison (reference doc, required) ---
fig, ax = plt.subplots(figsize=(10, 6))
for name, returns in all_net_returns.items():
    growth = (1 + returns.dropna()).cumprod()
    ax.plot(growth.index, growth.values, label=STRATEGY_LABELS[name], color=STRATEGY_COLORS[name], linewidth=2)

ax.set_yscale("log")
ax.set_title("Growth of $1 (net of transaction costs)")
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio value ($, log scale)")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "equity_curve_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Chart 2: factor performance breakdown (reference doc, required) ---
fig, axes = plt.subplots(2, 1, figsize=(10, 9))

# panel a: cumulative growth of each factor's own Fama-MacBeth return
for col in z_cols:
    growth = (1 + factor_returns_df[col].dropna()).cumprod()
    axes[0].plot(growth.index, growth.values, label=col.replace("z_", ""), linewidth=2)
axes[0].set_title("Cumulative factor return (Fama-MacBeth regression slope)")
axes[0].set_ylabel("Growth of $1 exposure")
axes[0].legend()

# panel b: IC t-stat per factor, with the conventional |t| = 2 significance
# threshold marked -- makes section 9's "none individually significant"
# finding visible directly on the chart, not just in a printed table
factor_names = [c.replace("z_", "") for c in ic_quality.index]
bar_colors = ["#2ca02c" if abs(t) >= 2 else "#d62728" for t in ic_quality["t_stat"]]
axes[1].bar(factor_names, ic_quality["t_stat"], color=bar_colors)
axes[1].axhline(2, color="black", linestyle="--", linewidth=1)
axes[1].axhline(-2, color="black", linestyle="--", linewidth=1)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("IC t-stat per factor (dashed lines = conventional significance threshold)")
axes[1].set_ylabel("t-stat")

fig.tight_layout()
fig.savefig(RESULTS_DIR / "factor_performance_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Chart 3: weight allocation over time ---
fig, axes = plt.subplots(2, 1, figsize=(10, 9), sharex=True)
factor_display_names = [c.replace("z_", "") for c in z_cols]

axes[0].stackplot(weights_df.index, [weights_df[c] for c in z_cols], labels=factor_display_names)
axes[0].set_title("IC-weighted: factor weight allocation over time")
axes[0].set_ylabel("Weight")
axes[0].legend(loc="upper left", ncol=4, fontsize=8)

axes[1].stackplot(kalman_weights_df.index, [kalman_weights_df[c] for c in z_cols], labels=factor_display_names)
axes[1].set_title("Kalman-weighted: factor weight allocation over time")
axes[1].set_ylabel("Weight")
axes[1].set_xlabel("Date")

fig.tight_layout()
fig.savefig(RESULTS_DIR / "weight_allocation_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Chart 4: underwater / drawdown chart ---
fig, ax = plt.subplots(figsize=(10, 5))
for name, returns in all_net_returns.items():
    growth = (1 + returns.dropna()).cumprod()
    drawdown = growth / growth.cummax() - 1
    ax.fill_between(drawdown.index, drawdown.values, 0, color=STRATEGY_COLORS[name], alpha=0.15)
    ax.plot(drawdown.index, drawdown.values, color=STRATEGY_COLORS[name], label=STRATEGY_LABELS[name], linewidth=1.5)

ax.set_title("Drawdown from running peak (net of transaction costs)")
ax.set_ylabel("Drawdown")
ax.set_xlabel("Date")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "drawdown_underwater.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Chart 5: long-short vs. long-only comparison ---
strategy_display = [STRATEGY_LABELS[s] for s in SCORE_COLS]
long_only_sharpe = [results_table.loc[f"{s}_net", "sharpe_ratio"] for s in SCORE_COLS]
long_short_sharpe = [long_short_results_table.loc[f"{s}_net", "sharpe_ratio"] for s in SCORE_COLS]

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(SCORE_COLS))
width = 0.35
ax.bar(x - width / 2, long_only_sharpe, width, label="Long-only (net)", color="#1f77b4")
ax.bar(x + width / 2, long_short_sharpe, width, label="Long-short (net)", color="#d62728")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(strategy_display)
ax.set_ylabel("Sharpe ratio")
ax.set_title("Long-only vs. long-short Sharpe ratio, net of costs")
ax.legend()

fig.tight_layout()
fig.savefig(RESULTS_DIR / "long_short_vs_long_only.png", dpi=150, bbox_inches="tight")
plt.show()